# Phase space

{py:func}`.phase_space` generates unweighted $n$-body phase space for arbitrary masses with EvtGen's phase space routine. In contrast to {py:class}`.Generator`, it needs no particle or decay table, so it can be called with different masses in one process and next to a {py:class}`.Generator`.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("svg")

In [ ]:
from pyevtgen import phase_space

momenta = phase_space(3.0969, [0.1350, 0.1396, 0.4937], n=200_000, seed=0)
momenta.shape

The result has shape `(n_events, n_daughters, 4)` with four-momenta $(E, p_x, p_y, p_z)$ in the rest frame of the parent. Unweighted phase space is uniform over the Dalitz plot:

In [ ]:
def mass_squared(p: np.ndarray) -> np.ndarray:
    return p[..., 0] ** 2 - np.sum(p[..., 1:] ** 2, axis=-1)


s01 = mass_squared(momenta[:, 0] + momenta[:, 1])
s12 = mass_squared(momenta[:, 1] + momenta[:, 2])
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.hist2d(s01, s12, bins=80, cmin=1, rasterized=True)
ax.set_xlabel("$m_{01}^2$ [GeV$^2$]")
ax.set_ylabel("$m_{12}^2$ [GeV$^2$]")
plt.show()

## Number of daughters

The routine supports 2 to 15 daughters, but it uses accept-reject sampling whose efficiency drops steeply with the number of daughters. The table below shows the generation rate for pions from a parent of 3 GeV. Beyond about nine daughters, generation becomes impractically slow, and a running call cannot be interrupted.

In [ ]:
n_daughters = np.arange(2, 9)
rates = []
for n in n_daughters:
    n_events = 2000
    start = time.perf_counter()
    phase_space(3.0, [0.14] * n, n=n_events, seed=0)
    rates.append(n_events / (time.perf_counter() - start))

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(n_daughters, rates, "o-")
ax.set_xlabel("number of daughters")
ax.set_ylabel("events per second")
ax.set_yscale("log")
plt.show()

For four or more daughters, see {doc}`four-body`.

Invalid input raises a {py:class}`ValueError`, for instance when the daughters are heavier than the parent:

In [ ]:
try:
    phase_space(1.0, [0.6, 0.6], n=1)
except ValueError as error:
    print(error)